# The curvature range from Hessian-vector products

The lesson reads the safe learning rate off the Hessian's eigenvalues. A model with a
million weights has a Hessian with $10^{12}$ entries, and nobody forms it. What you can
afford is a Hessian-vector product $H\mathbf{v}$, which for least squares costs two
matrix-vector products with $X$. Here you find the largest and the smallest curvature from
those products alone, then test the largest against an experiment: the learning rate at
which gradient descent starts to blow up.

**How to work through it.** Each part names one function to write. Fill in its
body where it says `TODO` and delete the `raise NotImplementedError` line, then run
the check cell below it. A correct function prints `Part N passed`. Otherwise the
check stops and says what went wrong.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n, d = 400, 5
X = rng.standard_normal((n, d)) * np.array([3.0, 1.5, 1.0, 0.5, 0.15])   # five features, five scales
noise = rng.normal(0, 0.1, n)
y_wide = 2.0 * X[:, 0] + noise          # a target driven by the widest feature
y_narrow = 6.0 * X[:, 4] + noise        # a target driven by the narrowest feature

def make_hvp(X):
    """v -> H v for the MSE of a linear model, H = (2/n) X^T X, without forming H."""
    m = len(X)
    return lambda v: 2 / m * (X.T @ (X @ v))

hvp = make_hvp(X)
print("X", X.shape, "  H v for v = e_1:", np.round(hvp(np.eye(d)[0]), 3))

## Part 1 (coding task): The largest curvature

Write `top_curvature(hvp, d, iters=300, seed=0)`, which returns the largest eigenvalue of
the symmetric positive semidefinite matrix behind `hvp`, a function that takes a vector of
length `d` and returns $H\mathbf{v}$. Use power iteration (see Eigenvalues and Spectral
Theory), starting from `np.random.default_rng(seed).standard_normal(d)`, and return the
Rayleigh quotient $\mathbf{v}^\top H\mathbf{v}$ of the final unit vector. You may call `hvp`
but never build $H$.

In [ ]:
def top_curvature(hvp, d, iters=300, seed=0):
    v = np.random.default_rng(seed).standard_normal(d)
    # TODO: power iteration with hvp, then the Rayleigh quotient.
    raise NotImplementedError

In [ ]:
top = top_curvature(hvp, d)
H = 2 / n * X.T @ X                     # formed here only to check the answer
lam = np.linalg.eigvalsh(H)
assert np.isfinite(top), f"got {top}: rescale v to length 1 every iteration, or its length overflows"
assert abs(top - lam[-1]) <= 1e-8 * lam[-1], f"got {top:.6f}, the largest eigenvalue is {lam[-1]:.6f}"
D = np.diag([1.0, 5.0, 2.0])
assert abs(top_curvature(lambda v: D @ v, 3) - 5.0) < 1e-8, "on diag(1, 5, 2) the answer should be 5"
print(f"largest curvature {top:.4f}, so gradient descent needs eta < {2 / top:.4f}")
print("Part 1 passed")

## Part 2 (coding task): The smallest curvature

Power iteration finds only the eigenvalue of largest size. Write
`bottom_curvature(hvp, d, top, iters=1000, seed=0)`, which returns the smallest eigenvalue
of $H$, given `top`, its largest. Use `top_curvature` on a different operator, built from
`hvp` and `top`, whose largest eigenvalue tells you the smallest of $H$.

In [ ]:
def bottom_curvature(hvp, d, top, iters=1000, seed=0):
    # TODO: build an operator from hvp and top, and use top_curvature on it.
    raise NotImplementedError

In [ ]:
lam = np.linalg.eigvalsh(2 / n * X.T @ X)      # the true eigenvalues, to check against
bottom = bottom_curvature(hvp, d, lam[-1])
assert not abs(bottom - lam[-1]) < 1e-3 * lam[-1], "this is the largest eigenvalue again: power iteration on H itself always finds the top"
assert not abs(bottom - (lam[-1] - lam[0])) < 1e-3 * lam[-1], "this is top minus the smallest eigenvalue, the answer of the shifted operator: convert it back"
assert abs(bottom - lam[0]) <= 1e-6 * lam[0], f"got {bottom:.6g}, the smallest eigenvalue is {lam[0]:.6g}"
D = np.diag([4.0, 0.5, 2.0])
assert abs(bottom_curvature(lambda v: D @ v, 3, 4.0) - 0.5) < 1e-8, "on diag(4, 0.5, 2) the answer should be 0.5"
kappa = lam[-1] / bottom
print(f"smallest curvature {bottom:.5f}, condition number {kappa:.1f}")
print(f"eta = 1/top needs about 7 * kappa = {7 * kappa:.0f} steps to cut the error 1,000-fold")
print("Part 2 passed")

## Part 3 (coding task): Measure where gradient descent blows up

Write `stability_edge(X, y, lo, hi, steps=400, tol=1e-4)`. A run with learning rate
`eta` starts from $\theta = \mathbf{0}$ and takes `steps` steps of gradient descent on the
mean squared error $\tfrac1n\lVert\mathbf{y} - X\theta\rVert^2$. Call the run diverged
when its loss turns non-finite or passes $10^{12}$ times the starting loss at any step, or
when it ends with a larger loss than it started with. Assume
runs at `lo` converge and runs at `hi` diverge, and find the boundary by bisection: test
the midpoint, keep the half that still has a converging end and a diverging end, and stop
when `hi - lo <= tol * hi`. Return the midpoint of the last interval.

The check runs it on both targets from the setup, with `lo = 0` and `hi = 1`, and
compares the edge with $2/\lambda_{\max}$.

In [ ]:
def stability_edge(X, y, lo, hi, steps=400, tol=1e-4):
    # TODO: a function that runs gradient descent and reports divergence, then bisection.
    raise NotImplementedError

In [ ]:
edges = {name: stability_edge(X, t, 0.0, 1.0) for name, t in (("wide", y_wide), ("narrow", y_narrow))}
y_huge = y_wide.copy()
y_huge[5] = 1e150                      # one reading off the scale
edges["huge"] = stability_edge(X, y_huge, 0.0, 1.0)
limit = 2 / np.linalg.eigvalsh(2 / n * X.T @ X)[-1]
for name, e in edges.items():
    print(f"target {name:6}: measured edge {e:.5f}   2 / largest curvature {limit:.5f}   ratio {e / limit:.4f}")
r_wide, r_narrow = edges["wide"] / limit, edges["narrow"] / limit
r_huge = edges["huge"] / limit
assert not (abs(r_wide - 1) < 0.02 and r_narrow > 1.5), "right for one target, far too high for the other: a run whose loss falls at first can still diverge later, so judge it by the loss after all the steps"
assert not (r_wide > 1.5 and r_narrow > 1.5), "both edges are far too high: check that a diverging midpoint lowers hi rather than raising lo"
assert r_huge < 1.5, (
    f"the huge-target edge is {r_huge:.1f} times the limit: the start loss is already 2.5e297, "
    "so 1e12 times it is inf, and the run's loss overflows to inf or nan, neither of which is greater than "
    "inf. Only a non-finite loss counts as diverged, so a run that has blown up looks converged "
    "without that test")
assert abs(r_wide - 1) < 0.02 and abs(r_narrow - 1) < 0.02, "the measured edge should be within 2% of 2 / largest curvature for both targets"
assert abs(r_huge - 1) < 0.02, "the huge reading scales one row but not the curvature, so the edge should still match 2 / largest curvature"
print("Part 3 passed")

## Solution

### Part 1: The largest curvature

In [ ]:
def top_curvature(hvp, d, iters=300, seed=0):
    v = np.random.default_rng(seed).standard_normal(d)
    v = v / np.linalg.norm(v)
    for _ in range(iters):
        w = hvp(v)
        v = w / np.linalg.norm(w)        # rescale every time, or the length overflows
    return float(v @ hvp(v))

**Why it works.** Write the start vector in the eigenvectors of $H$. Each product multiplies the part along
eigenvector $i$ by $\lambda_i$, so after $k$ products the part along the top eigenvector
has grown by $(\lambda_1/\lambda_i)^k$ relative to each other part, and the unit vector
turns toward it. Here $\lambda_1/\lambda_2 \approx 3.6$, so 300 iterations are far more
than enough. The Rayleigh quotient of a unit vector close to an eigenvector is that
eigenvalue, with an error that shrinks like the square of the vector's error. Without
rescaling, the length grows like $15.6^k$ and overflows to `inf` within a few hundred
iterations.

### Part 2: The smallest curvature

In [ ]:
def bottom_curvature(hvp, d, top, iters=1000, seed=0):
    flipped = lambda v: top * v - hvp(v)          # eigenvalues top - lambda_i, all >= 0
    return top - top_curvature(flipped, d, iters, seed)

**Why it works.** $H$ and $tI - H$, with $t$ the top eigenvalue, have the same eigenvectors, and an
eigenvalue $\lambda_i$ of $H$ becomes $t - \lambda_i$, which is at least 0. The largest
of these is $t - \lambda_{\min}$, so power iteration on $tI - H$ finds the flattest
direction, and subtracting from $t$ gives $\lambda_{\min}$. It needs more iterations than
part 1, because the two largest shifted eigenvalues, $t - 0.047$ and $t - 0.487$, are only
about 3% apart, and each iteration gains a factor of only about 1.03 on the competitor.
The condition number of about 330 explains why gradient descent on these raw features is
slow.

### Part 3: Measure where gradient descent blows up

In [ ]:
def stability_edge(X, y, lo, hi, steps=400, tol=1e-4):
    m = len(y)

    def diverges(eta):
        theta = np.zeros(X.shape[1])
        start = np.mean(y ** 2)                      # the loss at theta = 0
        for _ in range(steps):
            theta = theta + eta * 2 / m * X.T @ (y - X @ theta)
            loss = np.mean((y - X @ theta) ** 2)
            if not np.isfinite(loss) or loss > 1e12 * start:
                return True                          # stop before the numbers overflow
        return bool(loss > start)

    while hi - lo > tol * hi:
        mid = (lo + hi) / 2
        if diverges(mid):
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

**Why it works.** Along the top eigenvector each step multiplies the error by $1 - \eta\lambda_{\max}$, whose
size passes 1 at $\eta = 2/\lambda_{\max}$. Just above it the factor is only a little
beyond $-1$, so 400 steps are needed to make the growth visible, which is why the measured
edge sits a fraction of a percent above $2/\lambda_{\max}$. The two targets differ in where
the starting error lies. For `y_narrow` it lies almost entirely along the flattest feature,
so the first steps lower the loss even at $\eta$ several times the limit, while a tiny
component along the steep direction grows until it dominates. A test that looks only at the
first step, or at a few steps, puts the edge far too high there. Overflow is the other trap:
a diverging run reaches `inf` and then `nan`, and every comparison with `nan` is false, so
it has to be caught explicitly.